# 使用Logistic Regression对错误crop样本进行排除

采用人工标注的数据进行拟合LR，并以优化目标找不同的threshold apply到未知数据。

此过程可以较为**保守的排除**来进行多轮次循环，可以通过每轮训练新的线性头来进一步加强模型对错误样本的自信，以此达到保证排除noise（positive）强化recall的同时提升precision（减少误杀）。同时轮次内可以人工介入对排除样本进行人工筛选。

## 数据加载

In [2]:
import os
import re
import sqlite3
import sys
from pathlib import Path
from typing import Any

import gradio as gr
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"Project root not found from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline import utils
config = utils.load_pipeline_config()
DB_PATH = Path(utils.join_data_root(config['path']['db_path']))
utils.init_db(DB_PATH)
DATA_ROOT = DB_PATH.parent
DB_PATH, DATA_ROOT

(PosixPath('/Users/yukun/projects/wakareeru/data/commons_image_manifest.sqlite'),
 PosixPath('/Users/yukun/projects/wakareeru/data'))

In [3]:
features = pd.read_csv(DATA_ROOT / "demo_loss_feature.csv")


with sqlite3.connect(DB_PATH) as conn:
    SQLquery = '''
    SELECT id AS crop_id, noise_review_label FROM crops
    WHERE  noise_review_label IS NOT NULL
    
    '''
    metas = pd.read_sql_query(SQLquery, conn)
features.head(5)

,crop_id,mean,std,count,error_rate,pred_label,pred_label_rate,loss_tail_mean,label,loss_mean_pct_in_label
0,1,0.556498,0.573375,30,0.066667,E6系,0.933333,0.267969,E6系,0.581560
1,2,0.271206,0.422669,30,0.033333,E231系,0.966667,0.050488,E231系,0.218354
2,3,0.121574,0.413180,30,0.000000,E233系,1.000000,0.006866,E233系,0.173333
3,4,0.179531,0.299574,30,0.000000,E231系,1.000000,0.040101,E231系,0.034810
4,5,0.690713,0.846680,30,0.066667,E657系,0.933333,0.182793,E657系,0.686275


In [4]:
feature_columns = ['mean', 'loss_tail_mean', 'loss_mean_pct_in_label',"error_rate", 'pred_label_rate']

reviewed_data = pd.merge(left=metas, right=features, on='crop_id', how='inner')
print(f"共有{len(reviewed_data)}条已审核数据")

noise_positive_label = ['wrong_label']
excluding_label = ['bad_crop', "out_of_label_space"]

filtered = reviewed_data[~reviewed_data['noise_review_label'].isin(excluding_label)]
filtered['noise_review_label'] = filtered['noise_review_label'].apply(lambda x: 1 if x in noise_positive_label else 0).astype(int)
filtered.head(5)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    filtered[feature_columns],
    filtered['noise_review_label'],
    test_size=0.2,
    random_state=42,
    stratify=filtered['noise_review_label']
)
# holdout test集，以免在小样本上过拟合，无法评估模型在新数据上的表现



model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        class_weight="balanced",
        random_state=42,
        max_iter=1000,
    ),
)



cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

prob = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
threshold = 0.7
pred = (prob >= threshold).astype(int)

# print("n =", len(filtered), "positive =", int(y_train.sum()), "n_splits =", 4)
# print("ROC AUC:", roc_auc_score(y_train, prob))
# print("Average precision:", average_precision_score(y_train, prob))
confusion_matrix(y_train, pred)





共有387条已审核数据


/var/folders/h4/flmszp1906l3vn21tj84nvrr0000gn/T/ipykernel_57011/2740648441.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered['noise_review_label'] = filtered['noise_review_label'].apply(lambda x: 1 if x in noise_positive_label else 0).astype(int)
/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/sklearn/linear_model/_lo

array([[249,  16],
       [  7,  18]])

## 参数搜索

首先测试threshold的表现，以约束优化来找最优

In [5]:
thresholds = np.linspace(0.05, 0.95, 19)
clean_recall_min = 0.9
best_threshold = 0
max_noise_recall = 0
for threshold in thresholds:
    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

    prob = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
    pred = (prob >= threshold).astype(int)
    confusion_matrix(y_train, pred)
    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
    clean_recall = tn / (tn + fp)
    if clean_recall >= clean_recall_min:
        noise_recall = tp / (tp + fn)
        if noise_recall > max_noise_recall:
            max_noise_recall = noise_recall
            best_threshold = threshold
print(f"Best threshold: {best_threshold:.2f}, Noise recall: {max_noise_recall:.4f}")
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

prob = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
pred = (prob >= best_threshold).astype(int)
print(classification_report(y_train, pred,))
print(roc_auc_score(y_train, prob))

Best threshold: 0.60, Noise recall: 0.7600
              precision    recall  f1-score   support

           0       0.98      0.91      0.94       265
           1       0.44      0.76      0.56        25

    accuracy                           0.90       290
   macro avg       0.71      0.83      0.75       290
weighted avg       0.93      0.90      0.91       290

0.9311698113207547


/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed 

In [6]:
# 在测试集上评估模型性能
model.fit(X_train, y_train)
test_pred = (model.predict_proba(X_test)[:, 1] > best_threshold).astype(int)
print(classification_report(y_test, test_pred))


              precision    recall  f1-score   support

           0       0.97      0.84      0.90        67
           1       0.27      0.67      0.38         6

    accuracy                           0.82        73
   macro avg       0.62      0.75      0.64        73
weighted avg       0.91      0.82      0.85        73



/opt/anaconda3/envs/wakareeru/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
